# Inference Notebook — Abdou (Transformer + Classical ML)

Load pre-trained models and run predictions without retraining.

**Models available:**
- 11 classical ML models (top 3: ExtraTrees, GradientBoosting, RandomForest)
- 3 Transformer models: ChemBERTa-MLM, ChemBERTa-MTR, ChemBERTa-DL

## 1. Imports

In [ ]:
import pandas as pd, numpy as np, pickle, os, warnings, math
from rdkit import Chem
from rdkit.Chem import rdFingerprintGenerator, Descriptors
from rdkit.ML.Descriptors import MoleculeDescriptors
from sklearn.feature_selection import VarianceThreshold
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification
warnings.filterwarnings('ignore')

device = torch.device('mps' if torch.backends.mps.is_available() else 'cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

## 2. Load Data

In [ ]:
df = pd.read_csv('data/curated_data_rdkit_Abdou.csv')
df.rename(columns={'logC50':'log_LC50','SMILES_curated':'SMILES'}, inplace=True)
df['mol'] = df['SMILES'].apply(lambda s: Chem.MolFromSmiles(s) if s else None)
df = df.dropna(subset=['mol']).reset_index(drop=True)
df['toxic'] = (df['log_LC50'] > 1.0).astype(int)
print(f'Molecules: {len(df)} | Toxic: {df["toxic"].sum()} ({df["toxic"].mean()*100:.1f}%)')
df.head(3)

## 3. Load Saved Classical ML Models

In [ ]:
with open('models/classical_models_Abdou.pkl', 'rb') as f:
    cp = pickle.load(f)

trained_models = cp['models']
top3 = cp['top3']
sel = cp['preprocessor']['sel']
scaler = cp['preprocessor']['scaler']
feat_names = cp['preprocessor']['feat_names']
needs_scaling = cp['preprocessor']['needs_scaling']

print('Loaded classical models:', list(trained_models.keys()))
print(f'Top 3: {top3}')
print(f'Features: {len(feat_names)}')

### 3.1 Reproduce Test Set Results

In [ ]:
from sklearn.metrics import (accuracy_score, roc_auc_score, precision_score,
                             recall_score, f1_score)

gen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)
fps = np.array([gen.GetFingerprint(m) for m in df['mol']])
desc_names = [d[0] for d in Descriptors._descList]
calculator = MoleculeDescriptors.MolecularDescriptorCalculator(desc_names)
descs = np.array([list(calculator.CalcDescriptors(m)) for m in df['mol']])
X_raw = np.hstack([fps, descs])
y = df['toxic'].values
X = sel.transform(X_raw)

train_idx, test_idx, y_train, y_test = train_test_split(
    np.arange(len(y)), y, test_size=0.2, random_state=42, stratify=y)
X_train, X_test = X[train_idx], X[test_idx]
X_test_s = scaler.transform(X_test)
print(f'Test set: {len(X_test)} molecules')

results = []
for name, m in trained_models.items():
    use_s = name in needs_scaling
    X_te = X_test_s if use_s else X_test
    if hasattr(m, 'predict_proba'):
        proba = m.predict_proba(X_te)[:, 1]
    else:
        proba = m.decision_function(X_te)
    pred = (proba > 0.5).astype(int)
    results.append({'model': name,
        'auc_roc': roc_auc_score(y_test, proba),
        'accuracy': accuracy_score(y_test, pred),
        'precision': precision_score(y_test, pred, zero_division=0),
        'recall': recall_score(y_test, pred, zero_division=0),
        'f1': f1_score(y_test, pred, zero_division=0),
        'specificity': recall_score(1-y_test, 1-pred, zero_division=0)})

dfr = pd.DataFrame(results).sort_values('auc_roc', ascending=False).reset_index(drop=True)
print('=== Classical Benchmark Results ===')
print(dfr[['model','auc_roc','accuracy','f1','precision','recall','specificity']].to_string(index=False))

### 3.2 Predict on New SMILES (Classical Models)

In [ ]:
def predict_classical(smiles_list, model_name=None):
    mols = [Chem.MolFromSmiles(s) for s in smiles_list]
    valid = [m is not None for m in mols]
    if not all(valid):
        print(f'Warning: {sum(1 for v in valid if not v)} invalid SMILES')
    gen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)
    fps = np.array([gen.GetFingerprint(m) for m in mols if m is not None])
    calculator = MoleculeDescriptors.MolecularDescriptorCalculator(desc_names)
    descs = np.array([list(calculator.CalcDescriptors(m)) for m in mols if m is not None])
    X_new = sel.transform(np.hstack([fps, descs]))
    models_to_use = [model_name] if model_name else top3
    rows = []
    for name in models_to_use:
        if name not in trained_models: continue
        m = trained_models[name]
        use_s = name in needs_scaling
        X_te = scaler.transform(X_new) if use_s else X_new
        proba = m.predict_proba(X_te)[:, 1]
        for i, smi in enumerate([s for s, v in zip(smiles_list, valid) if v]):
            rows.append({'SMILES': smi, 'Model': name,
                         'Toxicity_Probability': round(proba[i], 4),
                         'Prediction': 'Toxic' if proba[i] > 0.5 else 'Non-toxic'})
    return pd.DataFrame(rows)

example_smiles = ['CCO', 'c1ccccc1', 'O=C(O)c1ccccc1', 'CCCCCCCCBr']
predict_classical(example_smiles)

---
## 4. Transformer Models

Load ChemBERTa models with trained classification heads.

### 4.1 Define Classification Head

In [ ]:
class SimpleHead(nn.Module):
    def __init__(self, hd):
        super().__init__()
        self.net = nn.Sequential(nn.Dropout(0.3), nn.Linear(hd,1))
    def forward(self, x):
        return self.net(x[:, 0, :])  # CLS token

### 4.2 Load Transformer Models
Downloads weights from HuggingFace (~300 MB each, cached locally).

In [ ]:
import pickle

transformer_configs = [
    ('MLM', 'DeepChem/ChemBERTa-77M-MLM', 'models/ChemBERTa_MLM_binary.pth'),
    ('MTR', 'DeepChem/ChemBERTa-77M-MTR', 'models/ChemBERTa_MTR_binary.pth'),
    ('Druglike', 'Derify/ChemBERTa-druglike', 'models/ChemBERTa_DL_binary.pth'),
]

loaded_transformers = {}
for short_name, hf_id, pth_path in transformer_configs:
    print(f'Loading {short_name} ({hf_id})...')
    tok = AutoTokenizer.from_pretrained(hf_id)
    model = AutoModelForSequenceClassification.from_pretrained(
        hf_id, num_labels=1, problem_type='regression', ignore_mismatched_sizes=True)
    model.classifier = SimpleHead(model.config.hidden_size)
    
    state = torch.load(pth_path, map_location=device, weights_only=True)
    model.load_state_dict(state, strict=False)
    model = model.to(device)
    model.eval()
    
    loaded_transformers[short_name] = {'model': model, 'tokenizer': tok}
    print(f'  Done')

print(f'\nLoaded: {list(loaded_transformers.keys())}')

### 4.3 Reproduce Transformer Test Set Results

In [ ]:
smiles_train, smiles_test = df.iloc[train_idx]['SMILES'].tolist(), df.iloc[test_idx]['SMILES'].tolist()
y_train_tensor = torch.tensor(y_train, dtype=torch.float)
y_test_tensor = torch.tensor(y_test, dtype=torch.float)

class SMILESDataset(Dataset):
    def __init__(self, sm, y, tok, ml=256):
        self.enc = tok(sm, padding='max_length', truncation=True, max_length=ml, return_tensors='pt')
        self.y = y
    def __len__(self): return len(self.y)
    def __getitem__(self, i): return {k: v[i] for k,v in self.enc.items()}, self.y[i]

def collate_smiles(b):
    return {k: torch.stack([b[i][0][k] for i in range(len(b))]) for k in b[0][0]}, \
           torch.stack([b[i][1] for i in range(len(b))])

transformer_results = {}
for name, cfg in loaded_transformers.items():
    model, tok = cfg['model'], cfg['tokenizer']
    te_ds = SMILESDataset(smiles_test, y_test_tensor, tok)
    te_loader = DataLoader(te_ds, 16, shuffle=False, collate_fn=collate_smiles)
    
    preds, targets = [], []
    with torch.no_grad():
        for b in te_loader:
            x = {k: v.to(device) for k,v in b[0].items()}
            y = b[1]
            logits = model(**x).logits
            preds.extend(torch.sigmoid(logits).squeeze(-1).cpu().numpy())
            targets.extend(y.cpu().numpy())
    preds = np.array(preds); targets = np.array(targets)
    binary = (preds > 0.5).astype(int)
    transformer_results[name] = {
        'accuracy': accuracy_score(targets, binary),
        'auc_roc': roc_auc_score(targets, preds),
        'sensitivity': recall_score(targets, binary, zero_division=0),
        'specificity': recall_score(1-targets, 1-binary, zero_division=0),
        'precision': precision_score(targets, binary, zero_division=0),
        'f1': f1_score(targets, binary, zero_division=0)}
    name_map = {'MLM': 'ChemBERTa-MLM', 'MTR': 'ChemBERTa-MTR', 'Druglike': 'ChemBERTa-DL'}
    r = transformer_results[name]
    print(f'{name_map[name]:18s}: AUC={r["auc_roc"]:.4f}  Acc={r["accuracy"]:.4f}  F1={r["f1"]:.4f}')

# Combined table
rows = []
for n in top3:
    r = [x for x in results if x['model'] == n][0]
    rows.append({'model': f'Classical: {n}',
        'auc_roc': r['auc_roc'], 'accuracy': r['accuracy'],
        'precision': r['precision'], 'recall': r['recall'],
        'specificity': r['specificity'], 'f1': r['f1']})
name_map = {'MLM': 'ChemBERTa-MLM', 'MTR': 'ChemBERTa-MTR', 'Druglike': 'ChemBERTa-DL'}
for n, r in transformer_results.items():
    rows.append({'model': name_map[n],
        'auc_roc': r['auc_roc'], 'accuracy': r['accuracy'],
        'precision': r['precision'], 'recall': r.get('sensitivity'),
        'specificity': r['specificity'], 'f1': r['f1']})
pdf = pd.DataFrame(rows).sort_values('auc_roc', ascending=False).reset_index(drop=True)
print('\n=== Transformer vs Classical Comparison ===')
print(pdf.to_string(index=False))

### 4.4 Predict on New SMILES (Transformer Models)

In [ ]:
def predict_transformer(smiles_list, model_name='MLM'):
    if model_name not in loaded_transformers:
        print(f'Available: {list(loaded_transformers.keys())}')
        return
    model = loaded_transformers[model_name]['model']
    tok = loaded_transformers[model_name]['tokenizer']
    
    enc = tok(smiles_list, padding=True, truncation=True, max_length=256, return_tensors='pt')
    enc = {k: v.to(device) for k,v in enc.items()}
    
    with torch.no_grad():
        logits = model(**enc).logits
        probs = torch.sigmoid(logits).squeeze(-1).cpu().numpy()
    
    name_map = {'MLM': 'ChemBERTa-MLM', 'MTR': 'ChemBERTa-MTR', 'Druglike': 'ChemBERTa-DL'}
    display_name = name_map.get(model_name, model_name)
    data = []
    for smi, prob in zip(smiles_list, probs):
        data.append({'SMILES': smi, 'Model': display_name,
            'Toxicity_Probability': round(float(prob), 4),
            'Prediction': 'Toxic' if prob > 0.5 else 'Non-toxic'})
    return pd.DataFrame(data)

# Example
predict_transformer(['CCO', 'c1ccccc1', 'O=C(O)c1ccccc1', 'CCCCCCCCBr'], 'MLM')

---
## 5. Saved Results (for Thesis)

All performance metrics saved as CSV files in `data/`.

In [ ]:
print('Classical benchmark:')
print(pd.read_csv('data/classical_benchmark.csv').to_string(index=False))
print('\nTransformer comparison:')
print(pd.read_csv('data/transformer_comparison.csv').to_string(index=False))